# ProtoSSL User study analysis

This notebook conducts:

- **Primary analysis:** participant-level paired comparison of the proportion of responses rated as good for ProtoSSL vs ProtoECGNet, done separately for the two tasks.
- **Primary test:** two-sided **Wilcoxon signed-rank test** across participants.
- **Comparative A/B/Both/Neither question:** descriptive summaries
- **Inter-rater agreement:** **Fleiss' kappa** for the binary yes/no ratings, reported overall and by label.

The **participants** are the primary unit of inference.


In [15]:
import math
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, ttest_rel, binomtest, t as tdist
from statsmodels.stats.inter_rater import fleiss_kappa

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


In [16]:
results = pd.read_csv("results.csv")
metadata = pd.read_csv("metadata.csv")

print("results shape:", results.shape)
print("metadata shape:", metadata.shape)
results.head(2)


results shape: (7, 131)
metadata shape: (20, 15)


,record_id,redcap_survey_identifier,user_study_form_timestamp,consent,prototypes_quality_choices,prototypea_quality,prototypeb_quality,prototypes_quality_choices_2,explanation_a,explanation_b,case1_prototypes_quality_choices,case1_prototypea_quality,case1_prototypeb_quality,case1_prototypes_quality_choices_2,case1_explanation_a,case1_explanation_b,case2_prototypes_quality_choices,case2_prototypea_quality,case2_prototypeb_quality,case2_prototypes_quality_choices_2,case2_explanation_a,case2_explanation_b,case3_prototypes_quality_choices,case3_prototypea_quality,case3_prototypeb_quality,case3_prototypes_quality_choices_2,case3_explanation_a,case3_explanation_b,case4_prototypes_quality_choices,case4_prototypea_quality,case4_prototypeb_quality,case4_prototypes_quality_choices_2,case4_explanation_a,case4_explanation_b,case5_prototypes_quality_choices,case5_prototypea_quality,case5_prototypeb_quality,case5_prototypes_quality_choices_2,case5_explanation_a,case5_explanation_b,case6_prototypes_quality_choices,case6_prototypea_quality,case6_prototypeb_quality,case6_prototypes_quality_choices_2,case6_explanation_a,case6_explanation_b,case7_prototypes_quality_choices,case7_prototypea_quality,case7_prototypeb_quality,case7_prototypes_quality_choices_2,case7_explanation_a,case7_explanation_b,case8_prototypes_quality_choices,case8_prototypea_quality,case8_prototypeb_quality,case8_prototypes_quality_choices_2,case8_explanation_a,case8_explanation_b,case9_prototypes_quality_choices,case9_prototypea_quality,case9_prototypeb_quality,case9_prototypes_quality_choices_2,case9_explanation_a,case9_explanation_b,case10_prototypes_quality_choices,case10_prototypea_quality,case10_prototypeb_quality,case10_prototypes_quality_choices_2,case10_explanation_a,case10_explanation_b,case11_prototypes_quality_choices,case11_prototypea_quality,case11_prototypeb_quality,case11_prototypes_quality_choices_2,case11_explanation_a,case11_explanation_b,case12_prototypes_quality_choices,case12_prototypea_quality,case12_prototypeb_quality,case12_prototypes_quality_choices_2,case12_explanation_a,case12_explanation_b,case13_prototypes_quality_choices,case13_prototypea_quality,case13_prototypeb_quality,case13_prototypes_quality_choices_2,case13_explanation_a,case13_explanation_b,case14_prototypes_quality_choices,case14_prototypea_quality,case14_prototypeb_quality,case14_prototypes_quality_choices_2,case14_explanation_a,case14_explanation_b,case15_prototypes_quality_choices,case15_prototypea_quality,case15_prototypeb_quality,case15_prototypes_quality_choices_2,case15_explanation_a,case15_explanation_b,case16_prototypes_quality_choices,case16_prototypea_quality,case16_prototypeb_quality,case16_prototypes_quality_choices_2,case16_explanation_a,case16_explanation_b,case17_prototypes_quality_choices,case17_prototypea_quality,case17_prototypeb_quality,case17_prototypes_quality_choices_2,case17_explanation_a,case17_explanation_b,case18_prototypes_quality_choices,case18_prototypea_quality,case18_prototypeb_quality,case18_prototypes_quality_choices_2,case18_explanation_a,case18_explanation_b,case19_prototypes_quality_choices,case19_prototypea_quality,case19_prototypeb_quality,case19_prototypes_quality_choices_2,case19_explanation_a,case19_explanation_b,case20_prototypes_quality_choices,case20_prototypea_quality,case20_prototypeb_quality,case20_prototypes_quality_choices_2,case20_explanation_a,case20_explanation_b,user_study_form_complete
0,5,NaN,2026-04-06 18:18:56,1,2,0,1,2,0,1,3,1,1,3,1,1,1,1,1,1,1,1,2,1,1,2,1,1,3,1,1,1,1,1,3,1,1,3,1,1,3,1,1,3,1,1,3,1,1,3,1,1,3,1,1,3,1,1,3,1,1,1,1,1,2,1,1,2,1,1,2,1,1,3,1,1,2,1,1,3,1,1,3,1,1,3,1,1,3,1,1,3,1,1,3,1,1,3,1,1,2,0,1,2,0,1,3,1,1,1,1,1,2,0,1,2,0,1,3,1,1,3,1,1,3,1,1,2,1,1,2
1,7,NaN,2026-04-09 11:26:53,1,2,0,1,2,0,1,3,1,1,1,1,1,3,1,1,3,1,1,3,0,0,2,0,1,1,1,0,1,1,0,3,1,1,3,1,1,1,1,0,1,1,0,1,1,1,1,1,0,3,1,1,3,1,1,3,1,1,1,1,0,3,1,1,2,1,0,1,1,0,1,1,0,2,1,1,3,1,1,2,0,1,3,1,1,3,1,1,1,1,1,3,1,1,2,1,1,2,0,1,2,0,1,1,1,0,1,1,0,2,0,1,2,0,1,1,1,1,1,1,0,2,0

### Decode REDCap responses into analysis tables

`yesno` contains one row per participant × case × task × model for the binary yes/no questions.

`prefs` contains one row per participant × case × task for the A/B/Both/Neither comparative question, decoded back to the actual model identities using the metadata file.


In [17]:
case_map = (
    metadata.rename(
        columns={
            "Study Index": "case_id",
            "Label": "label",
            "ProtoSSL Assignment": "ssl_assignment",
            "ProtoECGNet Assignment": "ecg_assignment",
        }
    )[["case_id", "label", "ssl_assignment", "ecg_assignment"]]
    .copy()
)
case_map["case_id"] = case_map["case_id"].astype(int)

pref_code = {1: "A", 2: "B", 3: "Both", 4: "Neither"}

yes_rows = []
pref_rows = []

for _, row in results.iterrows():
    participant = int(row["record_id"])
    for case_id in range(1, 21):
        meta_row = case_map.loc[case_map["case_id"] == case_id].iloc[0]

        task_specs = [
            ("global",
             f"case{case_id}_prototypea_quality",
             f"case{case_id}_prototypeb_quality",
             f"case{case_id}_prototypes_quality_choices"),
            ("paired",
             f"case{case_id}_explanation_a",
             f"case{case_id}_explanation_b",
             f"case{case_id}_prototypes_quality_choices_2"),
        ]

        for task, a_col, b_col, pref_col in task_specs:
            for shown_letter, col in [("A", a_col), ("B", b_col)]:
                actual_model = "ProtoSSL" if meta_row["ssl_assignment"] == shown_letter else "ProtoECGNet"
                yes_rows.append(
                    {
                        "participant": participant,
                        "case_id": case_id,
                        "label": meta_row["label"],
                        "task": task,
                        "model": actual_model,
                        "good": int(row[col]),
                    }
                )

            pref_value = pref_code[int(row[pref_col])]
            if pref_value in ["A", "B"]:
                actual_pref = "ProtoSSL" if meta_row["ssl_assignment"] == pref_value else "ProtoECGNet"
            else:
                actual_pref = pref_value

            pref_rows.append(
                {
                    "participant": participant,
                    "case_id": case_id,
                    "label": meta_row["label"],
                    "task": task,
                    "preference": actual_pref,
                }
            )

yesno = pd.DataFrame(yes_rows)
prefs = pd.DataFrame(pref_rows)

print("yesno shape:", yesno.shape)
print("prefs shape:", prefs.shape)
yesno.head()


yesno shape: (560, 6)
prefs shape: (280, 5)


,participant,case_id,label,task,model,good
0,5,1,AMI,global,ProtoSSL,1
1,5,1,AMI,global,ProtoECGNet,1
2,5,1,AMI,paired,ProtoSSL,1
3,5,1,AMI,paired,ProtoECGNet,1
4,5,2,AMI,global,ProtoSSL,1


### Descriptive summaries for the binary yes/no questions

In [18]:
overall_yesno = (
    yesno.groupby(["task", "model"])["good"]
    .agg(n_yes="sum", n_total="count", proportion="mean")
    .reset_index()
)
overall_yesno


,task,model,n_yes,n_total,proportion
0,global,ProtoECGNet,93,140,0.6643
1,global,ProtoSSL,128,140,0.9143
2,paired,ProtoECGNet,95,140,0.6786
3,paired,ProtoSSL,116,140,0.8286


In [19]:
label_yesno = (
    yesno.groupby(["task", "label", "model"])["good"]
    .agg(n_yes="sum", n_total="count", proportion="mean")
    .reset_index()
)
label_yesno


,task,label,model,n_yes,n_total,proportion
0,global,AMI,ProtoECGNet,23,35,0.6571
1,global,AMI,ProtoSSL,30,35,0.8571
2,global,CLBBB,ProtoECGNet,33,35,0.9429
3,global,CLBBB,ProtoSSL,32,35,0.9143
4,global,CRBBB,ProtoECGNet,19,35,0.5429
5,global,CRBBB,ProtoSSL,32,35,0.9143
6,global,PVC,ProtoECGNet,18,35,0.5143
7,global,PVC,ProtoSSL,34,35,0.9714
8,paired,AMI,ProtoECGNet,23,35,0.6571
9,paired,AMI,ProtoSSL,28,35,0.8000


### Primary analysis

For each participant and each task, compute the proportion of responses rated as good for each model across the 20 cases. Then compare ProtoSSL vs ProtoECGNet with a **paired Wilcoxon signed-rank test**.

Because there are **two primary task-level hypotheses** (`global` and `paired`), apply **Holm correction** across those two p-values.


In [20]:
participant_summary = (
    yesno.groupby(["participant", "task", "model"])["good"]
    .mean()
    .unstack("model")
    .reset_index()
)

participant_summary["difference"] = (
    participant_summary["ProtoSSL"] - participant_summary["ProtoECGNet"]
)

participant_summary


model,participant,task,ProtoECGNet,ProtoSSL,difference
0,5,global,0.9000,1.0000,0.1000
1,5,paired,0.9000,1.0000,0.1000
2,7,global,0.7000,0.8000,0.1000
3,7,paired,0.6000,0.8000,0.2000
4,8,global,0.4500,0.8000,0.3500
5,8,paired,0.5000,0.5500,0.0500
6,9,global,0.6000,0.9500,0.3500
7,9,paired,0.5500,0.9000,0.3500
8,10,global,0.6000,0.9500,0.3500
9,10,paired,0.7500,0.9500,0.2000


In [21]:
def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    m = len(pvalues)
    order = np.argsort(pvalues)
    adjusted = np.empty_like(pvalues)
    running_max = 0.0

    for rank, idx in enumerate(order):
        candidate = (m - rank) * pvalues[idx]
        running_max = max(running_max, candidate)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted

def participant_level_analysis(df):
    rows = []
    raw_wilcoxon_p = []

    for task in ["global", "paired"]:
        sub = df[df["task"] == task].copy()
        diffs = sub["difference"].to_numpy()

        w = wilcoxon(diffs, alternative="two-sided", zero_method="wilcox", method="approx")
        ttest = ttest_rel(sub["ProtoSSL"], sub["ProtoECGNet"])
        sign = binomtest(np.sum(diffs > 0), np.sum(diffs != 0), p=0.5, alternative="two-sided")

        mean_diff = float(np.mean(diffs))
        sd_diff = float(np.std(diffs, ddof=1))
        se_diff = sd_diff / math.sqrt(len(diffs))
        tcrit = tdist.ppf(0.975, df=len(diffs) - 1)
        ci_low = mean_diff - tcrit * se_diff
        ci_high = mean_diff + tcrit * se_diff

        rows.append(
            {
                "task": task,
                "n_participants": len(sub),
                "ProtoSSL_mean": sub["ProtoSSL"].mean(),
                "ProtoECGNet_mean": sub["ProtoECGNet"].mean(),
                "mean_difference": mean_diff,
                "ci95_low": ci_low,
                "ci95_high": ci_high,
                "wilcoxon_W": float(w.statistic),
                "wilcoxon_p": float(w.pvalue),
                "paired_t_p": float(ttest.pvalue),
                "sign_test_p": float(sign.pvalue),
            }
        )
        raw_wilcoxon_p.append(float(w.pvalue))

    out = pd.DataFrame(rows)
    out["wilcoxon_p_holm"] = holm_adjust(out["wilcoxon_p"].to_numpy())
    return out

primary_results = participant_level_analysis(participant_summary)
primary_results


,task,n_participants,ProtoSSL_mean,ProtoECGNet_mean,mean_difference,ci95_low,ci95_high,wilcoxon_W,wilcoxon_p,paired_t_p,sign_test_p,wilcoxon_p_holm
0,global,7,0.9143,0.6643,0.2500,0.1432,0.3568,0.0000,0.0178,0.0012,0.0156,0.0355
1,paired,7,0.8286,0.6786,0.1500,0.0501,0.2499,0.0000,0.0178,0.0104,0.0156,0.0355


## Per-label participant-level summaries

These are useful to show **where** the overall pattern comes from, but I recommend keeping them **descriptive only** in the paper because each label has only 5 cases.


In [22]:
participant_by_label = (
    yesno.groupby(["participant", "task", "label", "model"])["good"]
    .mean()
    .unstack("model")
    .reset_index()
)
participant_by_label["difference"] = (
    participant_by_label["ProtoSSL"] - participant_by_label["ProtoECGNet"]
)

per_label_summary = (
    participant_by_label.groupby(["task", "label"])
    .agg(
        ProtoSSL_mean=("ProtoSSL", "mean"),
        ProtoECGNet_mean=("ProtoECGNet", "mean"),
        mean_difference=("difference", "mean"),
    )
    .reset_index()
)
per_label_summary


,task,label,ProtoSSL_mean,ProtoECGNet_mean,mean_difference
0,global,AMI,0.8571,0.6571,0.2000
1,global,CLBBB,0.9143,0.9429,-0.0286
2,global,CRBBB,0.9143,0.5429,0.3714
3,global,PVC,0.9714,0.5143,0.4571
4,paired,AMI,0.8000,0.6571,0.1429
5,paired,CLBBB,0.7714,0.9429,-0.1714
6,paired,CRBBB,0.8571,0.5429,0.3143
7,paired,PVC,0.8857,0.5714,0.3143


### Descriptive summaries for the comparative A/B/Both/Neither question

In [23]:
preference_overall = (
    prefs.groupby(["task", "preference"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
preference_overall


preference,task,Both,Neither,ProtoECGNet,ProtoSSL
0,global,51,0,25,64
1,paired,34,6,36,64


In [24]:
preference_by_label = (
    prefs.groupby(["task", "label", "preference"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
preference_by_label


preference,task,label,Both,Neither,ProtoECGNet,ProtoSSL
0,global,AMI,16,0,5,14
1,global,CLBBB,17,0,10,8
2,global,CRBBB,8,0,5,22
3,global,PVC,10,0,5,20
4,paired,AMI,8,3,9,15
5,paired,CLBBB,12,2,17,4
6,paired,CRBBB,8,0,5,22
7,paired,PVC,6,1,5,23


### Fleiss' kappa for the binary yes/no ratings


In [25]:
def fleiss_from_yesno(df):
    table = []
    for _, g in df.groupby(["case_id", "task", "model"]):
        counts = g["good"].value_counts().reindex([0, 1], fill_value=0)
        table.append(counts.values)

    table = np.asarray(table)
    return float(fleiss_kappa(table))

overall_kappa = fleiss_from_yesno(yesno)

kappa_by_label = (
    yesno.groupby("label", group_keys=False)
    .apply(fleiss_from_yesno)
    .rename("fleiss_kappa")
    .reset_index()
)

print("Overall Fleiss' kappa:", round(overall_kappa, 4))
kappa_by_label


Overall Fleiss' kappa: 0.2877


/var/folders/9j/f0qlzhxj2klgf3bxm77sqz300000gn/T/ipykernel_34866/3937073973.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(fleiss_from_yesno)


,label,fleiss_kappa
0,AMI,0.2023
1,CLBBB,0.0542
2,CRBBB,0.3000
3,PVC,0.4000


### Fleiss' kappa for the A/B/Both/Neither ratings


In [26]:
from statsmodels.stats.inter_rater import fleiss_kappa
import numpy as np

def fleiss_from_pref(df):
    categories = ["ProtoSSL", "ProtoECGNet", "Both", "Neither"]
    table = []
    for _, g in df.groupby(["case_id", "task"]):
        counts = g["preference"].value_counts().reindex(categories, fill_value=0)
        table.append(counts.values)

    table = np.asarray(table)
    return float(fleiss_kappa(table))

overall_kappa_pref = fleiss_from_pref(prefs)

kappa_by_label_pref = (
    prefs.groupby("label", group_keys=False)
    .apply(fleiss_from_pref)
    .rename("fleiss_kappa")
    .reset_index()
)

print("Overall Fleiss' kappa (A/B/Both/Neither):", round(overall_kappa_pref, 4))
kappa_by_label_pref

Overall Fleiss' kappa (A/B/Both/Neither): 0.1953


/var/folders/9j/f0qlzhxj2klgf3bxm77sqz300000gn/T/ipykernel_34866/259028057.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(fleiss_from_pref)


,label,fleiss_kappa
0,AMI,0.2170
1,CLBBB,0.0174
2,CRBBB,0.1322
3,PVC,0.1425


### Compact tables for manuscript drafting

In [27]:
primary_results_rounded = primary_results.copy()
for col in ["ProtoSSL_mean", "ProtoECGNet_mean", "mean_difference", "ci95_low", "ci95_high", "wilcoxon_p", "wilcoxon_p_holm", "paired_t_p", "sign_test_p"]:
    primary_results_rounded[col] = primary_results_rounded[col].round(4)
primary_results_rounded


,task,n_participants,ProtoSSL_mean,ProtoECGNet_mean,mean_difference,ci95_low,ci95_high,wilcoxon_W,wilcoxon_p,paired_t_p,sign_test_p,wilcoxon_p_holm
0,global,7,0.9143,0.6643,0.2500,0.1432,0.3568,0.0000,0.0178,0.0012,0.0156,0.0355
1,paired,7,0.8286,0.6786,0.1500,0.0501,0.2499,0.0000,0.0178,0.0104,0.0156,0.0355


In [28]:
label_yesno_pivot = label_yesno.copy()
label_yesno_pivot["summary"] = (
    label_yesno_pivot["n_yes"].astype(str)
    + "/"
    + label_yesno_pivot["n_total"].astype(str)
    + " ("
    + (100 * label_yesno_pivot["proportion"]).round(1).astype(str)
    + "%)"
)
label_yesno_pivot = (
    label_yesno_pivot[["task", "label", "model", "summary"]]
    .pivot(index=["task", "label"], columns="model", values="summary")
    .reset_index()
)
label_yesno_pivot


model,task,label,ProtoECGNet,ProtoSSL
0,global,AMI,23/35 (65.7%),30/35 (85.7%)
1,global,CLBBB,33/35 (94.3%),32/35 (91.4%)
2,global,CRBBB,19/35 (54.3%),32/35 (91.4%)
3,global,PVC,18/35 (51.4%),34/35 (97.1%)
4,paired,AMI,23/35 (65.7%),28/35 (80.0%)
5,paired,CLBBB,33/35 (94.3%),27/35 (77.1%)
6,paired,CRBBB,19/35 (54.3%),30/35 (85.7%)
7,paired,PVC,20/35 (57.1%),31/35 (88.6%)
